# Week 48, Day 2: String matching and KMP

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w048d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
text = "aaaaaaaaab"
pattern = "aaab"

comparisons = 0
found = []
for i in range(len(text) - len(pattern) + 1):
    j = 0
    while j < len(pattern):
        comparisons += 1
        if text[i + j] != pattern[j]:
            break
        j += 1
    if j == len(pattern):
        found.append(i)
print(found, comparisons)

In [ ]:
def prefix_function(s):
    pi = [0] * len(s)
    k = 0                                  # length of the current border
    for i in range(1, len(s)):
        while k > 0 and s[i] != s[k]:
            k = pi[k - 1]                  # fall back to the next shorter border
        if s[i] == s[k]:
            k += 1
        pi[i] = k
    return pi

for s in ["ababaca", "aaaa", "abcab", "aabaaab"]:
    print(f"{s:<8} {prefix_function(s)}")

In [ ]:
def prefix_function(s):
    pi, k = [0] * len(s), 0
    for i in range(1, len(s)):
        while k > 0 and s[i] != s[k]:
            k = pi[k - 1]
        if s[i] == s[k]:
            k += 1
        pi[i] = k
    return pi

def kmp_find_all(text, pattern):
    m = len(pattern)
    if m == 0:
        return []
    pi, j, found = prefix_function(pattern), 0, []
    for i, c in enumerate(text):
        while j > 0 and c != pattern[j]:
            j = pi[j - 1]
        if c == pattern[j]:
            j += 1
        if j == m:
            found.append(i - m + 1)
            j = pi[m - 1]
    return found

print(kmp_find_all("abababcabab", "abab"))
print(kmp_find_all("aaaaaaaaab", "aaab"))

In [ ]:
def prefix_function(s):
    pi, k = [0] * len(s), 0
    for i in range(1, len(s)):
        while k > 0 and s[i] != s[k]:
            k = pi[k - 1]
        if s[i] == s[k]:
            k += 1
        pi[i] = k
    return pi

for s in ["abcabcabc", "abcabca", "aaaa", "abcd"]:
    p = len(s) - prefix_function(s)[-1]
    print(f"{s:<10} period {p}, repetition of {s[:p]!r}: {len(s) % p == 0}")

pattern, text = "aba", "abababa"
combined = prefix_function(pattern + "#" + text)
print([i - 2 * len(pattern) for i, v in enumerate(combined) if v == len(pattern)])

> **Key idea.** The naive search tries every position: O(n × m) worst case, because it forgets what earlier comparisons showed. The prefix function gives, for each position, the length of the longest proper prefix that is also a suffix (a border); it's built in O(m) by falling back through shorter borders with `k = pi[k - 1]`. KMP scans the text once, falling back the same way on a mismatch and after each match: O(n + m), never re-reading the text, so it works on streams. The prefix function also gives all borders and the shortest period, `len(s) - pi[-1]`. In everyday code, use `in`, `find` and `re`.

### Exercise: The obvious method

Write `naive_find_all(text, pattern)`: every starting position of `pattern` in `text` (overlaps allowed), comparing one character at a time, and returning `(positions, comparisons)`, where `comparisons` counts every character comparison made. An empty pattern gives `([], 0)`. No `find`, `in` on strings, slicing comparisons or `re`.

In [ ]:
def naive_find_all(text, pattern):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_naive():\n    \"positions and counts\"\n    assert naive_find_all(\"aaaaaaaaab\", \"aaab\") == ([6], 28)\n    assert naive_find_all(\"abababa\", \"aba\") == ([0, 2, 4], 11)\n    assert naive_find_all(\"abc\", \"\") == ([], 0) and naive_find_all(\"ab\", \"abc\") == ([], 0)\n    rng = random.Random(1)\n    for _ in range(200):\n        t = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 20)))\n        p = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 4)))\n        assert naive_find_all(t, p)[0] == [i for i in range(len(t) - len(p) + 1) if t[i:i + len(p)] == p]")

### Exercise: The prefix function

Write `prefix_function(s)` in O(len(s)), for any sequence (strings, lists of words). The checks compare with a slow definition and time it on a 200,000-character string of repeated `a`s with a `b` at the end.

In [ ]:
def prefix_function(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow(s):\n    out = []\n    for i in range(len(s)):\n        t = s[:i + 1]\n        out.append(max([k for k in range(len(t)) if t[:k] == t[len(t) - k:]], default=0))\n    return out\ndef test_prefix():\n    \"values\"\n    assert prefix_function(\"ababaca\") == [0, 0, 1, 2, 3, 0, 1] and prefix_function(\"\") == []\n    assert prefix_function([\"to\", \"be\", \"or\", \"to\", \"be\"]) == [0, 0, 0, 1, 2]\n    rng = random.Random(2)\n    for _ in range(300):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 14)))\n        assert prefix_function(s) == slow(s), s\ndef test_speed():\n    \"linear\"\n    s = \"a\" * 200_000 + \"b\"\n    t0 = time.perf_counter()\n    pi = prefix_function(s)\n    assert pi[-2] == 199_999 and pi[-1] == 0 and time.perf_counter() - t0 < 2")

### Exercise: KMP, one character at a time

Write `Matcher(pattern)` (a non-empty pattern) with `feed(c)`: take the next character of a stream and return `True` if a match of the pattern ends with it. Keep only the pattern, its prefix function and `j`; never store the text. Then `kmp_find_all(text, pattern)` using a `Matcher`, and `count_in_stream(chars, pattern)` for any iterable of characters. The checks count comparisons: under 4(n + m) on any input, where the naive method would need about n × m.

In [ ]:
class Matcher:
    def __init__(self, pattern):
        ___

def kmp_find_all(text, pattern):
    ___

def count_in_stream(chars, pattern):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nclass C(str):\n    count = 0\n    def __ne__(self, other):\n        C.count += 1\n        return str.__ne__(self, other)\n    def __eq__(self, other):\n        C.count += 1\n        return str.__eq__(self, other)\n    __hash__ = str.__hash__\ndef test_kmp():\n    \"matches\"\n    assert kmp_find_all(\"abababcabab\", \"abab\") == [0, 2, 7] and kmp_find_all(\"aaaaaaaaab\", \"aaab\") == [6]\n    assert kmp_find_all(\"\", \"a\") == [] and kmp_find_all(\"abc\", \"abcd\") == []\n    m = Matcher(\"aa\")\n    assert [m.feed(c) for c in \"aaab\"] == [False, True, True, False]\n    assert count_in_stream(iter(\"abababa\"), \"aba\") == 3\n    rng = random.Random(3)\n    for _ in range(300):\n        t = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 30)))\n        p = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 5)))\n        assert kmp_find_all(t, p) == [i for i in range(len(t) - len(p) + 1) if t[i:i + len(p)] == p]\ndef test_linear():\n    \"never backs up\"\n    for text, pat in [(\"a\" * 3000 + \"b\", \"a\" * 50 + \"b\"), (\"ab\" * 2000, \"abab\" * 10 + \"c\")]:\n        C.count = 0\n        kmp_find_all([C(ch) for ch in text], [C(ch) for ch in pat])\n        assert C.count < 4 * (len(text) + len(pat)), C.count")

### Exercise: Periods and borders

Using a prefix function (write one), write `borders(s)`: the lengths of every border of `s` (proper prefixes that are also suffixes), longest first; `shortest_period(s)`; and `as_repetition(s)`: a pair `(unit, times)` with the shortest `unit` such that `unit * times == s` (`(s, 1)` if there's none shorter; `("", 0)` for the empty string).

In [ ]:
def prefix_function(s):
    ___

def borders(s):
    ___

def shortest_period(s):
    ___

def as_repetition(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_period():\n    \"structure\"\n    assert borders(\"abacaba\") == [3, 1] and borders(\"aaaa\") == [3, 2, 1] and borders(\"abc\") == []\n    assert shortest_period(\"abcabca\") == 3 and shortest_period(\"aaaa\") == 1 and shortest_period(\"abcd\") == 4\n    assert as_repetition(\"abcabcabc\") == (\"abc\", 3) and as_repetition(\"abcabca\") == (\"abcabca\", 1) and as_repetition(\"\") == (\"\", 0)\n    rng = random.Random(4)\n    for _ in range(300):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 12)))\n        assert borders(s) == [k for k in range(len(s) - 1, 0, -1) if s[:k] == s[-k:]]\n        p = min(p for p in range(1, len(s) + 1) if all(s[i] == s[i + p] for i in range(len(s) - p)))\n        assert shortest_period(s) == p")

### Exercise: Matching more than characters

KMP works on any sequence whose items can be compared. Write `find_phrase(text, phrase)`: the word positions where the words of `phrase` occur in a row in `text`, comparing words case-insensitively and ignoring punctuation at either end of a word (strip `.,;:!?"'` from each word, then lower-case it). Use a prefix function over the lists of cleaned words.

In [ ]:
def prefix_function(s):
    ___

def find_phrase(text, phrase):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_phrase():\n    \"phrases\"\n    text = \"To be, or not to be: that is the question. To be honest, TO BE sure!\"\n    assert find_phrase(text, \"to be\") == [0, 4, 10, 13]\n    assert find_phrase(text, \"the question\") == [8] and find_phrase(text, \"be or not\") == [1]\n    assert find_phrase(text, \"missing words\") == [] and find_phrase(\"\", \"to\") == []")

### Bug hunt: Falling all the way back

On a mismatch, this prefix function falls straight back to 0 instead of to the next shorter border, so it misses borders: this prints `[0, 1, 0, 1, 2, 1, 0]`. Fall back with `k = pi[k - 1]` until the characters match, so it prints `[0, 1, 0, 1, 2, 2, 3]`.

In [ ]:
def prefix_function(s):
    pi, k = [0] * len(s), 0
    for i in range(1, len(s)):
        if k > 0 and s[i] != s[k]:
            k = 0
        if s[i] == s[k]:
            k += 1
        pi[i] = k
    return pi

print(prefix_function("aabaaab"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the next shorter border\"\n    expect_output(\"[0, 1, 0, 1, 2, 2, 3]\")\n    assert prefix_function(\"aabaab\") == [0, 1, 0, 1, 2, 3]")

### Bug hunt: Starting over after a match

After finding a match, this KMP resets `j` to 0, so a match that overlaps the one just found is missed: this prints `[0, 4]`. Fall back to the longest border instead, so it prints `[0, 2, 4]`.

In [ ]:
def prefix_function(s):
    pi, k = [0] * len(s), 0
    for i in range(1, len(s)):
        while k > 0 and s[i] != s[k]:
            k = pi[k - 1]
        if s[i] == s[k]:
            k += 1
        pi[i] = k
    return pi

def kmp_find_all(text, pattern):
    pi, j, found = prefix_function(pattern), 0, []
    for i, c in enumerate(text):
        while j > 0 and c != pattern[j]:
            j = pi[j - 1]
        if c == pattern[j]:
            j += 1
        if j == len(pattern):
            found.append(i - len(pattern) + 1)
            j = 0
    return found

print(kmp_find_all("abababa", "aba"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"overlaps\"\n    expect_output(\"[0, 2, 4]\")")

### Bug hunt: The last position

This naive search stops one position too early, so a match right at the end of the text is never tried: this prints `[0]`. Try every start from 0 to `len(text) - len(pattern)` inclusive, so it prints `[0, 4]`.

In [ ]:
def find_all(text, pattern):
    found = []
    for i in range(len(text) - len(pattern)):
        if text[i:i + len(pattern)] == pattern:
            found.append(i)
    return found

print(find_all("cat cat", "cat"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the end too\"\n    expect_output(\"[0, 4]\")")

### Puzzle: The prefix function

Arrange the lines into the prefix function. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the prefix function of "abcabd"
q = ___   # the prefix function of "aaaa"
r = ___   # the shortest period of "abaababaab"
s = ___   # every start of "aa" in "aaaa", overlaps allowed
t = ___   # "aaaa".count("aa")  (str.count doesn't overlap)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [0, 0, 0, 1, 2, 0] and q == [0, 1, 2, 3] and r == 5, \"p, q and r\"\n    assert s == [0, 1, 2] and t == 2, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the Z-function

A close relative of the prefix function is the **Z-function**: `z[i]` is the length of the longest common prefix of `s` and `s[i:]` (and `z[0]` is defined as `len(s)`). Computed naively it's O(n²); the linear method keeps a window `[l, r)`, the rightmost segment found so far that matches a prefix of `s`. For a new i inside the window, the value at `i - l` (already known) gives a head start: `z[i]` is at least `min(r - i, z[i - l])`; then extend by comparing characters, and move the window if it now reaches further right. Each comparison that succeeds moves `r` right, so the total is O(n). (The method is often credited to Dan Gusfield's 1997 textbook on string algorithms.)

Write `z_function(s)`, then use it for:

- `z_find_all(text, pattern)`: matches, via the Z-function of `pattern + "\0" + text` (`"\0"` won't appear in normal text);
- `distinct_prefix_counts(s)`: for every prefix length k from 1 to len(s), how many times the prefix `s[:k]` occurs in `s` (overlaps allowed), as a list. Build it in O(n) from the Z-function: count how many positions have each z value, then accumulate from long to short, since an occurrence of a long prefix is also one of every shorter prefix.

The checks compare with slow versions and time a 200,000-character string.

In [ ]:
def z_function(s):
    ___

def z_find_all(text, pattern):
    ___

def distinct_prefix_counts(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow_z(s):\n    out = []\n    for i in range(len(s)):\n        k = 0\n        while i + k < len(s) and s[k] == s[i + k]:\n            k += 1\n        out.append(k)\n    return out\ndef test_z():\n    \"values\"\n    assert z_function(\"aabxaab\") == [7, 1, 0, 0, 3, 1, 0] and z_function(\"\") == []\n    rng = random.Random(5)\n    for _ in range(300):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 15)))\n        assert z_function(s) == slow_z(s), s\n        p = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 3)))\n        assert z_find_all(s, p) == [i for i in range(len(s) - len(p) + 1) if s[i:i + len(p)] == p]\n        assert distinct_prefix_counts(s) == [sum(1 for i in range(len(s) - k + 1) if s[i:i + k] == s[:k]) for k in range(1, len(s) + 1)]\ndef test_speed():\n    \"linear\"\n    s = \"ab\" * 100_000\n    t0 = time.perf_counter()\n    z = z_function(s)\n    c = distinct_prefix_counts(s)\n    assert z[2] == 199_998 and c[0] == 100_000 and time.perf_counter() - t0 < 3")

## Warm-up

At every starting position the first three characters match before the fourth fails, so the same `a`s in the text get compared again and again. With a pattern of a thousand `a`s and a `b`, and a text of a million `a`s, that's about a billion comparisons.

## The obvious method

The **naive** (or brute-force) search tries each starting position `i` from 0 to n − m and compares `text[i + j]` with `pattern[j]` until a mismatch or a full match. It's O(n × m) in the worst case, though on everyday text a mismatch usually comes at the first or second character, so it's nearer O(n) in practice.

Its real waste is **forgetting**. In the warm-up, after matching `aaa` at position 0 and failing on the fourth character, the search already knows that `text[1:3]` is `aa`, yet it moves to position 1 and compares those characters all over again. KMP keeps that knowledge.

## The prefix function

Everything KMP knows comes from the pattern alone. For each position i of a string s, the **prefix function** `pi[i]` is the length of the longest **proper prefix** of `s[:i + 1]` that is also a **suffix** of it ("proper" means not the whole thing). Such a prefix-that-is-also-a-suffix is called a **border**.

```text
s:    a  b  a  b  a  c  a
i:    0  1  2  3  4  5  6
pi:   0  0  1  2  3  0  1

pi[4] = 3: in "ababa", "aba" is both a prefix and a suffix
pi[5] = 0: no proper prefix of "ababac" ends in "c"
pi[6] = 1: "ababaca" begins and ends with "a"
```

Computing it cleverly is the heart of the algorithm. Suppose you know `pi[i - 1] = k`: the first k characters match the k characters ending at i − 1. Now look at `s[i]`:

- If `s[i] == s[k]`, the border grows by one: `pi[i] = k + 1`.
- If not, try the **next shorter border** of the same prefix, which is `pi[k - 1]` (a border of a border is a border), and compare again. Keep falling back until a comparison succeeds or k reaches 0.

Why is it O(m), with a `while` inside a `for`? `k` goes up by at most one per step of the `for`, so over the whole run it rises at most m times, and each pass of the `while` lowers it. It can't fall more often than it rose, so the `while` runs fewer than m times **in total**. The same "count the total, not the worst single step" reasoning as amortised analysis in Week 41.

## Knuth, Morris and Pratt

KMP runs the same idea over the text. Keep `j`, the number of pattern characters currently matched. For each text character `c`:

- while `j > 0` and `c` doesn't match `pattern[j]`, fall back: `j = pi[j - 1]`. This slides the pattern along to the longest alignment still consistent with what's been seen, without ever re-reading the text;
- if `c` matches `pattern[j]`, `j += 1`;
- if `j == m`, a match ends here, at text position i, so it started at `i - m + 1`. Then fall back with `j = pi[m - 1]` so overlapping matches are found too.

The same argument as before makes the search O(n), so KMP is **O(n + m)** in all: O(m) to study the pattern, O(n) to scan the text. Just as important, it reads the text **once, left to right, never backing up**, so it works on a stream (a network connection, a file too big to load) one character at a time, which today's exercises use.

## What else the prefix function knows

Borders describe a string's repetitive structure, so the prefix function answers questions that have nothing to do with searching:

- **All borders** of s: `pi[-1]`, then `pi[pi[-1] - 1]`, and so on down to 0.
- **The shortest period** of s, the smallest p such that `s[i] == s[i + p]` everywhere: it's `len(s) - pi[-1]`. If p divides `len(s)`, s is a repetition of `s[:p]`; "abcabcabc" is "abc" three times.
- **Searching with one array.** Compute the prefix function of `pattern + "#" + text` (with `#` a character in neither); wherever the value equals m, a match ends there. This is KMP in a single function call.

## In real Python

For everyday searching, use the built-ins: `pattern in text`, `text.find(pattern)`, `text.count(pattern)` (non-overlapping), and `re` (Week 23) for anything fancier. They're written in C, and CPython's search combines several techniques, including (since Python 3.10) the **Two-Way** algorithm of Crochemore and Perrin (1991) for long patterns, which, like KMP, guarantees linear time. Write KMP yourself when you need what the built-ins don't give: matching a **stream** a character at a time, matching sequences that aren't strings (lists of tokens, events, DNA as lists), overlapping matches, or the prefix function's other answers.

## Practice

## Quiz

## Daily challenge

## Before you go

Compute the prefix function of your first name followed by itself (for example "adaada") by hand, then check it with code. Tomorrow: rolling hashes, going further than Week 44's Rabin and Karp to compare any two substrings in O(1).